# Week 3 - Data Preprocessing Deliverable

**Business objective:** prepare CRMLS SingleFamilyResidence sold data for ClosePrice prediction so pricing, listing strategy, and demand allocation decisions can be evaluated on the newest market month.

This notebook combines the team's Week 3 work:

1. missing-value handling
2. categorical-to-numeric encoding
3. numerical issue flags and scaling
4. time-based train/test split

## Source Notebook Coverage

| Source Notebook | Teammate work | Where it is included here | Notes |
|---|---|---|---|
| `02_preprocessing.ipynb` - Amelia| Load monthly files, CRMLS filter, missing-value summary, coordinate removal, median imputation, categorical missing handling, binary Y/N filling | Sections 2, 4, 5, 7 | Kept the logic, but avoided selecting only a small subset of columns because that would throw away valid modeling features. |
| `02_preprocessing_categorical_to_numerical.ipynb` - Amanda | Load/concat files, CRMLS filter, identify categorical fields, binary conversion, one-hot setup for `Levels` and `AssociationFeeFrequency`, date conversion | Sections 2, 3, 5, 7 | Replaced `ColumnTransformer` with direct pandas encoding so the notebook stays readable and has no helper functions. |
| `Rebecca_week3_numerical_preprocessing.ipynb` - Rebecca | Numerical summary, key numeric fields, outlier flags, strict numerical cleaning, scale-after-split rule | Sections 4, 5, 6, 7 | Kept review outliers as flags and fit scaling on train only to avoid leakage. |
| Week 3 requirement | Most recent month as test set; previous `X` months as train set; `X` is tunable | Section 6 | Default `X = 12`; candidate windows are exported for comparison. |

## Assumptions and Review Standard

- Scope is restricted to `PropertyType == "Residential"` and `PropertySubType == "SingleFamilyResidence"`.
- Target is `ClosePrice`.
- `CloseDate` is the transaction date used for the time-based split.
- The most recent available close month is the test set.
- The default training window is `X = 12` immediately preceding months. `X` is tunable; candidate row counts are reported below.
- Missing target or close date cannot support the prediction task, so those rows are removed.
- Missing latitude/longitude is removed because the project requires geographic segmentation and enrichment.
- Numerical hard failures are removed; review-level outliers are retained with flags.
- Scaling and imputation parameters are fit on train only, then applied to test to avoid leakage.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Imports, Paths, and Configuration

In [13]:
# Core libraries used in this notebook.
# pathlib is used for paths because it works cleanly on both local machines and Colab.
from pathlib import Path
import glob
import os
import re

import numpy as np
import pandas as pd

# Display settings make notebook outputs easier to inspect.
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

# The project data covers Jan 2025 through May 2026.
# We list the months explicitly so it is obvious which files are expected.
MONTHS = [
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512",
    "202601", "202602", "202603", "202604", "202605",
]

# Week 3 requires the most recent month as test and the previous X months as train.
# X is tunable. We use 12 as the default and also report other candidate windows.
TRAINING_WINDOW_MONTHS = 12
TRAINING_WINDOW_CANDIDATES = [3, 6, 9, 12, 15, 16]

# Prediction target and date field used for time-based evaluation.
TARGET_COL = "ClosePrice"
DATE_COL = "CloseDate"

# Numerical fields that are relevant for pricing or data-quality review.
# ClosePrice is included for summary checks but excluded from feature scaling later.
KEY_NUMERIC_COLS = [
    "ClosePrice", "ListPrice", "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "YearBuilt", "DaysOnMarket", "GarageSpaces", "ParkingTotal",
    "Latitude", "Longitude", "AssociationFee", "FireplacesTotal", "Stories",
]

# Binary listing attributes. These are amenity-like fields where values should become 0/1.
BINARY_COLS = [
    "ViewYN", "WaterfrontYN", "BasementYN", "PoolPrivateYN", "AttachedGarageYN",
    "FireplaceYN", "NewConstructionYN", "latfilled", "lonfilled",
]

# Low-cardinality categorical fields can be one-hot encoded without creating too many columns.
LOW_CARDINALITY_CATEGORICAL_COLS = [
    "Levels", "AssociationFeeFrequency", "StateOrProvince",
]

# Higher-cardinality geographic fields are frequency encoded later.
# This keeps useful market-location signal without creating hundreds of dummy columns.
FREQUENCY_CATEGORICAL_COLS = [
    "City", "PostalCode", "CountyOrParish", "MLSAreaMajor",
    "HighSchoolDistrict", "ElementarySchoolDistrict", "MiddleOrJuniorSchoolDistrict",
]

# These fields are identifiers, names, addresses, or agent/office fields.
# They are kept in the audit data but should not be treated as stable model features.
DROP_FROM_MODEL = [
    "ListingKey", "ListingKeyNumeric", "ListingId", "UnparsedAddress", "ListAgentEmail",
    "ListAgentFirstName", "ListAgentLastName", "ListAgentFullName",
    "CoListAgentFirstName", "CoListAgentLastName",
    "BuyerAgentFirstName", "BuyerAgentLastName", "BuyerAgentMlsId",
    "ListOfficeName", "BuyerOfficeName", "CoListOfficeName",
    "BuilderName", "SubdivisionName", "StreetNumberNumeric",
]


# Try common Colab Drive paths first, then local project paths.
# If the team runs this in Colab, the IDX Exchange path from the screenshots should work.
candidate_paths = [
    Path("/content/drive/MyDrive/IDX Summer Intern/california"),
]

DATA_PATH = None
for path in candidate_paths:
    if path.exists() and list(path.glob("CRMLSSold2025*.csv")):
        DATA_PATH = path
        break

# Stop early if the raw monthly files cannot be found.
# This is better than silently running on the wrong folder.
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find CRMLSSold monthly CSVs. Update DATA_PATH to the folder containing CRMLSSold202501.csv ... CRMLSSold202605.csv."
    )

# Output folder for all Week 3 deliverables.
OUTPUT_DIR = Path("/content/drive/MyDrive/IDX Summer Intern/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_PATH:", DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

DATA_PATH: /content/drive/MyDrive/IDX Summer Intern/california
OUTPUT_DIR: /content/drive/MyDrive/IDX Summer Intern/outputs


## 2. Load Monthly Sold Files and Apply CRMLS Scope Filter

In [14]:
# Each monthly CSV is loaded separately, then combined into one table.
# source_month records which file each row came from, which helps audit duplicates and time coverage.
frames = []
loaded_files = []
missing_files = []

for month in MONTHS:
    matches = sorted(DATA_PATH.glob(f"CRMLSSold{month}*.csv"))

    # If a month is missing, record it instead of failing immediately.
    # The notebook prints missing months so the team can decide if the data pull is incomplete.
    if not matches:
        missing_files.append(month)
        continue

    file_path = matches[0]
    temp = pd.read_csv(file_path, low_memory=False)
    temp["source_month"] = month
    frames.append(temp)
    loaded_files.append(file_path.name)

if not frames:
    raise ValueError("No monthly CRMLS sold files were loaded.")

# Combine all monthly data. sort=False preserves original columns without forcing alphabetical order.
df_raw = pd.concat(frames, ignore_index=True, sort=False)

print(f"Loaded {len(loaded_files)} files")
print("Missing months:", missing_files if missing_files else "none")
print("Raw shape:", df_raw.shape)

required = {"PropertyType", "PropertySubType"}
missing = required.difference(df_raw.columns)
if missing:
    raise KeyError(f"Missing required CRMLS filter columns: {missing}")

# Non-negotiable project scope:
# only Residential + SingleFamilyResidence records are valid for this pipeline.
df = df_raw[
    (df_raw["PropertyType"].astype(str).str.strip() == "Residential")
    & (df_raw["PropertySubType"].astype(str).str.strip() == "SingleFamilyResidence")
].copy()

print("Rows before filter:", len(df_raw))
print("Rows after Residential + SingleFamilyResidence filter:", len(df))
print("Rows removed by property filter:", len(df_raw) - len(df))
df.head()

Loaded 17 files
Missing months: none
Raw shape: (363970, 81)
Rows before filter: 363970
Rows after Residential + SingleFamilyResidence filter: 181482
Rows removed by property filter: 182488


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,PropertyType,LivingArea,ListPrice,DaysOnMarket,ListOfficeName,BuyerOfficeName,CoListOfficeName,ListAgentFullName,CoListAgentFirstName,CoListAgentLastName,BuyerAgentMlsId,BuyerAgentFirstName,BuyerAgentLastName,FireplacesTotal,AssociationFeeFrequency,AboveGradeFinishedArea,ListingKeyNumeric,MLSAreaMajor,TaxAnnualAmount,CountyOrParish,MlsStatus,ElementarySchool,AttachedGarageYN,ParkingTotal,BuilderName,PropertySubType,LotSizeAcres,SubdivisionName,BuyerOfficeAOR,YearBuilt,StreetNumberNumeric,ListingId,BathroomsTotalInteger,City,TaxYear,BuildingAreaTotal,BedroomsTotal,ContractStatusChangeDate,ElementarySchoolDistrict,CoBuyerAgentFirstName,PurchaseContractDate,ListingContractDate,BelowGradeFinishedArea,BusinessType,StateOrProvince,CoveredSpaces,MiddleOrJuniorSchool,FireplaceYN,Stories,HighSchool,Levels,LotSizeDimensions,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled,source_month
11,SanDiego,SanDiego,"Carpet,Tile",False,NaN,NaN,False,530000.0,1104130366,tom@structuresd.com,2025-01-31,530000.0,Tom,Damico,32.821023,-116.941739,1155 Pepper Dr,Residential,1656.0,530000.0,0,Structure Real Estate Inc.,Compass,NaN,Tom Damico,NaN,NaN,SAND-639005,Todd,Armstrong,NaN,NaN,NaN,1104130366,92021 - El Cajon,NaN,San Diego,Closed,NaN,NaN,4.0,NaN,SingleFamilyResidence,0.3168,El Cajon,SanDiego,1960.0,1155.0,250017852SD,2.0,El Cajon,NaN,NaN,4.0,2025-01-31,NaN,NaN,2025-01-17,2025-01-01,NaN,NaN,CA,NaN,NaN,False,1.0,NaN,One,NaN,13800.0,NaN,False,0.0,NaN,92021,NaN,13800.0,NaN,False,False,202501
12,SanDiego,SanDiego,NaN,False,NaN,NaN,False,700000.0,1104128744,chadtrower@gmail.com,2025-01-31,700000.0,Chad,Trower,32.826301,-117.103107,5174 Fino Dr,Residential,1558.0,700000.0,0,"eXp Realty of California, Inc.","eXp Realty of California, Inc.",NaN,Chad Trower,NaN,NaN,SAND-665130,Chad,Trower,NaN,Monthly,NaN,1104128744,92124 - Tierrasanta,NaN,San Diego,Closed,NaN,NaN,0.0,NaN,SingleFamilyResidence,0.0689,Tierrasanta,SanDiego,1972.0,5174.0,250017843SD,2.0,San Diego,NaN,NaN,4.0,2025-01-31,NaN,NaN,2025-01-31,2024-10-21,NaN,NaN,CA,NaN,NaN,False,1.0,NaN,One,NaN,3000.0,NaN,False,0.0,NaN,92124,170.0,3000.0,NaN,False,False,202501
13,SanDiego,SanDiego,"Carpet,Tile",False,NaN,NaN,False,1680000.0,1104128653,lisa@premiercoastalrealty.com,2025-01-31,1680000.0,Lisa,Bang,32.921240,-117.224366,4207 Calle Mar De Ballenas,Residential,2286.0,1680000.0,0,Premier Coastal Realty,Premier Coastal Realty,NaN,Lisa Bang,NaN,NaN,SAND-637524,Lisa,Bang,NaN,Monthly,NaN,1104128653,92130 - Carmel Valley,NaN,San Diego,Closed,NaN,True,4.0,NaN,SingleFamilyResidence,NaN,Carmel Valley,SanDiego,1999.0,4207.0,250017842SD,3.0,San Diego,NaN,NaN,4.0,2025-01-31,NaN,NaN,2025-01-09,2025-01-09,NaN,NaN,CA,NaN,NaN,True,2.0,NaN,Two,NaN,NaN,NaN,False,2.0,NaN,92130,26.0,NaN,NaN,False,False,202501
14,CaliforniaDesert,CaliforniaDesert,Tile,True,NaN,NaN,True,1195000.0,1104128077,lisa@traditionproperties.net,2025-01-31,1195000.0,Lisa,Putnam,33.691553,-116.306923,49195 Avenida El Nido,Residential,2695.0,1195000.0,0,"Double Eagle Real Estate, Inc.",California Lifestyle Realty,NaN,Lisa Putnam,NaN,NaN,CDAR-D73324,Kathy,Schowe,NaN,Monthly,NaN,1104128077,313 - La Quinta South of HWY 111,NaN,Riverside,Closed,NaN,True,3.0,NaN,SingleFamilyResidence,0.2700,LQCC Golf Estates,CaliforniaDesert,1990.0,49195.0,219123923DA,3.0,La Quinta,NaN,NaN,3.0,2025-01-31,NaN,NaN,2025-01-31,2025-01-01,NaN,NaN,CA,NaN,NaN,True,1.0,NaN,One,NaN,11761.0,NaN,False,3.0,NaN,92253,275.0,11761.0,NaN,False,False,202501
16,BeverlyHillsGreaterLa,BeverlyHillsGreaterLa,NaN,True,NaN,NaN,NaN,3800000.0,1104126716,Djacobson.re@gmail.com,2025-01-31,3800000.0,Daniel,Jacobson,34.136178,-118.432085,3620 Glenridge Drive,Residential,Na

## 3. Date Conversion, Required-Field Checks, and Duplicate Handling

Rows are not dropped blindly. The only required-field removals are for missing `ClosePrice` or `CloseDate`, because those rows cannot be used for supervised price prediction or time-based evaluation.

In [15]:
DATE_COLS = ["CloseDate", "ContractStatusChangeDate", "PurchaseContractDate", "ListingContractDate"]

# Convert date-like columns to datetime.
# errors="coerce" changes invalid date strings to NaT, making bad dates visible in missing checks.
for col in DATE_COLS:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# ClosePrice and CloseDate are required:
# - ClosePrice is the supervised learning target.
# - CloseDate is required for the time-based train/test split.
required_for_model = [TARGET_COL, DATE_COL]
for col in required_for_model:
    if col not in df.columns:
        raise KeyError(f"Missing required modeling column: {col}")

# Remove rows that cannot support the modeling task.
# This is a justified removal, not a blind drop.
before_required = len(df)
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df[df[TARGET_COL].notna() & df[DATE_COL].notna()].copy()
print("Rows removed for missing target or CloseDate:", before_required - len(df))

# Deduplicate listings.
# If ListingKey exists, it is the best available listing identifier.
# We keep the latest record by CloseDate/source_month because it is most likely to reflect final sale information.
before_dedup = len(df)
if "ListingKey" in df.columns:
    df = df.sort_values([DATE_COL, "source_month"], na_position="first")
    duplicate_count = df.duplicated("ListingKey", keep="last").sum()
    df = df.drop_duplicates("ListingKey", keep="last").copy()
    print("Duplicate ListingKey rows removed:", int(duplicate_count))
else:
    # Fallback only if ListingKey is unavailable.
    duplicate_count = df.duplicated().sum()
    df = df.drop_duplicates().copy()
    print("Exact duplicate rows removed:", int(duplicate_count))

print("Rows after de-duplication:", len(df), "| removed:", before_dedup - len(df))
df[[DATE_COL, TARGET_COL, "source_month"]].head()

Rows removed for missing target or CloseDate: 0
Duplicate ListingKey rows removed: 133
Rows after de-duplication: 181349 | removed: 133


,CloseDate,ClosePrice,source_month
5881,2025-01-01,3720000.0,202501
8435,2025-01-01,400000.0,202501
8721,2025-01-01,660595.0,202501
9488,2025-01-01,1325000.0,202501
10477,2025-01-01,980000.0,202501


## 4. Missing-Value and Numerical Summaries

In [16]:
# Missing-value summary for all columns.
# This tells the team which fields need imputation, removal, or exclusion.
missing_summary = (
    pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": df.isna().mean() * 100,
        "dtype": df.dtypes.astype(str),
    })
    .sort_values(["missing_percent", "missing_count"], ascending=False)
)

# Numerical summary for all numeric columns.
# Percentiles help identify impossible values and extreme outliers.
numerical_summary_rows = []
for col in df.select_dtypes(include=[np.number]).columns:
    s = pd.to_numeric(df[col], errors="coerce")
    q = s.quantile([0, .01, .05, .25, .5, .75, .95, .99, 1]).to_dict()
    numerical_summary_rows.append({
        "column": col,
        "count": int(s.count()),
        "missing_count": int(s.isna().sum()),
        "missing_pct": round(float(s.isna().mean() * 100), 4),
        "zero_count": int((s == 0).sum()),
        "negative_count": int((s < 0).sum()),
        "mean": s.mean(),
        "std": s.std(),
        "min": q.get(0),
        "p1": q.get(.01),
        "p5": q.get(.05),
        "p25": q.get(.25),
        "median": q.get(.5),
        "p75": q.get(.75),
        "p95": q.get(.95),
        "p99": q.get(.99),
        "max": q.get(1),
    })

numerical_summary = pd.DataFrame(numerical_summary_rows)

# Save summaries so the cleaning decisions are documented outside the notebook too.
missing_summary.to_csv(OUTPUT_DIR / "missing_summary.csv")
numerical_summary.to_csv(OUTPUT_DIR / "numerical_summary.csv", index=False)

display(missing_summary.head(20))
display(numerical_summary[numerical_summary["column"].isin(KEY_NUMERIC_COLS)])

,missing_count,missing_percent,dtype
FireplacesTotal,181349,100.000000,float64
AboveGradeFinishedArea,181349,100.000000,float64
TaxAnnualAmount,181349,100.000000,float64
TaxYear,181349,100.000000,float64
ElementarySchoolDistrict,181349,100.000000,float64
BusinessType,181349,100.000000,object
CoveredSpaces,181349,100.000000,float64
MiddleOrJuniorSchoolDistrict,181349,100.000000,float64
WaterfrontYN,181253,99.947063,object
BelowGradeFinishedArea,180084,99.302450,float64


,column,count,missing_count,missing_pct,zero_count,negative_count,mean,std,min,p1,p5,p25,median,p75,p95,p99,max
2,ClosePrice,181349,0,0.0000,1,0,1.334471e+06,7.538503e+06,0.000000,236000.000000,368000.000000,625000.000000,890000.000000,1.425000e+06,3.199000e+06,6.415000e+06,9.895000e+08
3,Latitude,181332,17,0.0094,22,2,3.472846e+01,1.749296e+00,-22.863239,32.671571,32.847805,33.759028,34.080416,3.475324e+01,3.796059e+01,3.973208e+01,4.378444e+01
4,Longitude,181332,17,0.0094,22,181296,-1.185957e+02,3.199188e+00,-124.193201,-122.446985,-122.186770,-119.136088,-118.023491,-1.172577e+02,-1.165485e+02,-1.162413e+02,3.290000e+02
5,LivingArea,181252,97,0.0535,82,0,2.047289e+03,1.045057e+03,0.000000,736.000000,966.000000,1385.000000,1816.000000,2.438000e+03,3.829000e+03,5.682490e+03,5.650000e+04
6,ListPrice,181349,0,0.0000,0,0,1.267488e+06,1.579754e+06,8000.000000,248686.240000,374999.000000,625000.000000,895000.000000,1.399998e+06,3.195000e+06,6.500000e+06,1.375000e+08
7,DaysOnMarket,181349,0,0.0000,7654,18,3.953712e+01,5.270642e+01,-265.000000,0.000000,1.000000,8.000000,19.000000,5.100000e+01,1.410000e+02,2.460000e+02,2.177000e+03
8,FireplacesTotal,0,181349,100.0000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,ParkingTotal,181348,1,0.0006,10702,29,3.081876e+00,4.599802e+01,-143.000000,0.000000,0.000000,2.000000,2.000000,3.000000e+00,6.000000e+00,1.100000e+01,1.572000e+04
14,YearBuilt,181228,121,0.0667,0,0,1.975862e+03,2.761543e+01,1776.000000,1910.000000,1926.000000,1956.000000,1976.000000,1.998000e+03,2.022000e+03,2.025000e+03,2.026000e+03
16,BathroomsTotalInteger,181329,20,0.0110,84,0,2.633136e+00,1.140968e+00,0.000000,1.000000,1.000000,2.000000,2.000000,3.000000e+00,5.000000e+00,6.000000e+00,4.500000e+01


## 5. Missing Values, Binary Encoding, and Numerical Quality Flags

Decision rule:

- Binary Y/N fields are converted to 0/1; missing is treated as 0 only for amenity-presence fields where missing means the listing did not report the feature as present.
- Hard numerical failures are removed because they corrupt model learning and stakeholder price interpretation.
- Review outliers are retained with flags because luxury, rural, or unusual properties can be valid.

In [17]:
# Work on a copy so the filtered raw table remains available for comparison if needed.
df_clean = df.copy()

# Convert binary fields to 0/1.
# Missing or unrecognized amenity values are treated as 0 because they are not reported as present.
binary_mapping = {
    "true": 1, "false": 0, "t": 1, "f": 0, "yes": 1, "no": 0,
    "y": 1, "n": 0, "1": 1, "0": 0, "nan": 0, "none": 0, "": 0,
}

for col in BINARY_COLS:
    if col in df_clean.columns:
        normalized = df_clean[col].astype(str).str.strip().str.lower()
        df_clean[col] = normalized.map(binary_mapping).fillna(0).astype(int)

# Convert important numeric fields to numeric Series before creating flags.
# If a column is missing from a future data pull, use an all-NaN Series so the code still runs.
close_price = pd.to_numeric(df_clean["ClosePrice"], errors="coerce")
list_price = pd.to_numeric(df_clean["ListPrice"], errors="coerce") if "ListPrice" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
living_area = pd.to_numeric(df_clean["LivingArea"], errors="coerce") if "LivingArea" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
bedrooms = pd.to_numeric(df_clean["BedroomsTotal"], errors="coerce") if "BedroomsTotal" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
bathrooms = pd.to_numeric(df_clean["BathroomsTotalInteger"], errors="coerce") if "BathroomsTotalInteger" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
lot_size = pd.to_numeric(df_clean["LotSizeSquareFeet"], errors="coerce") if "LotSizeSquareFeet" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
year_built = pd.to_numeric(df_clean["YearBuilt"], errors="coerce") if "YearBuilt" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
days_on_market = pd.to_numeric(df_clean["DaysOnMarket"], errors="coerce") if "DaysOnMarket" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
garage_spaces = pd.to_numeric(df_clean["GarageSpaces"], errors="coerce") if "GarageSpaces" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
parking_total = pd.to_numeric(df_clean["ParkingTotal"], errors="coerce") if "ParkingTotal" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
latitude = pd.to_numeric(df_clean["Latitude"], errors="coerce") if "Latitude" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
longitude = pd.to_numeric(df_clean["Longitude"], errors="coerce") if "Longitude" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)

# Price-to-list ratio is a useful sanity check for sale price consistency.
# Extremely tiny or huge ratios usually indicate data errors, not real market behavior.
price_ratio = close_price / list_price.replace(0, np.nan)
df_clean["ClosePrice_to_ListPrice_ratio"] = price_ratio

# Strict flags: likely data errors that should be removed before modeling.
df_clean["flag_closeprice_nonpositive"] = (close_price <= 0).fillna(False).astype(int)
df_clean["flag_price_ratio_extreme"] = ((price_ratio < 0.05) | (price_ratio > 5)).fillna(False).astype(int)

# Review flags: unusual values that might be valid for luxury or rural properties.
# These rows stay in the data, but the flags allow later error analysis by outlier type.
df_clean["flag_livingarea_extreme"] = ((living_area <= 0) | (living_area > 20000)).fillna(False).astype(int)
df_clean["flag_bedrooms_extreme"] = ((bedrooms < 0) | (bedrooms > 15)).fillna(False).astype(int)
df_clean["flag_bathrooms_extreme"] = ((bathrooms < 0) | (bathrooms > 15)).fillna(False).astype(int)
df_clean["flag_lotsize_extreme"] = ((lot_size <= 0) | (lot_size > 5_000_000)).fillna(False).astype(int)
df_clean["flag_yearbuilt_invalid"] = ((year_built < 1800) | (year_built > 2026)).fillna(False).astype(int)
df_clean["flag_daysonmarket_negative"] = (days_on_market < 0).fillna(False).astype(int)
df_clean["flag_garage_extreme"] = ((garage_spaces < 0) | (garage_spaces > 10)).fillna(False).astype(int)
df_clean["flag_parking_extreme"] = ((parking_total < 0) | (parking_total > 20)).fillna(False).astype(int)
df_clean["flag_latitude_outside_ca"] = ((latitude < 32) | (latitude > 42.5)).fillna(False).astype(int)
df_clean["flag_longitude_outside_ca"] = ((longitude < -125) | (longitude > -114)).fillna(False).astype(int)

# Strict issue = row should be removed.
# Review issue = row should be kept but monitored.
strict_flags = [
    "flag_closeprice_nonpositive", "flag_price_ratio_extreme",
    "flag_yearbuilt_invalid", "flag_daysonmarket_negative",
    "flag_latitude_outside_ca", "flag_longitude_outside_ca",
]
review_flags = [
    "flag_livingarea_extreme", "flag_bedrooms_extreme", "flag_bathrooms_extreme",
    "flag_lotsize_extreme", "flag_garage_extreme", "flag_parking_extreme",
]

df_clean["numeric_strict_issue_flag"] = df_clean[strict_flags].max(axis=1)
df_clean["numeric_review_flag"] = df_clean[review_flags].max(axis=1)

# Remove only strict numerical failures.
# Review outliers are not removed because expensive homes, large lots, or rural properties can be legitimate.
before_strict = len(df_clean)
df_clean = df_clean[df_clean["numeric_strict_issue_flag"] == 0].copy()
print("Rows removed by strict numerical rules:", before_strict - len(df_clean))

# Coordinates are required because the project includes geographic segmentation/enrichment.
# We remove missing coordinates instead of imputing fake locations.
if {"Latitude", "Longitude"}.issubset(df_clean.columns):
    before_coordinates = len(df_clean)
    df_clean = df_clean[df_clean["Latitude"].notna() & df_clean["Longitude"].notna()].copy()
    print("Rows removed for missing coordinates:", before_coordinates - len(df_clean))
else:
    print("Latitude/Longitude not both available; coordinate filtering skipped.")

flag_cols = [c for c in df_clean.columns if c.startswith("flag_")] + ["numeric_strict_issue_flag", "numeric_review_flag"]
display(df_clean[flag_cols].sum().sort_values(ascending=False))
print("Cleaned shape:", df_clean.shape)

Rows removed by strict numerical rules: 147
Rows removed for missing coordinates: 17


,0
numeric_review_flag,794
flag_parking_extreme,386
flag_lotsize_extreme,268
flag_garage_extreme,90
flag_livingarea_extreme,88
flag_bathrooms_extreme,18
flag_bedrooms_extreme,7
flag_price_ratio_extreme,0
flag_closeprice_nonpositive,0
flag_daysonmarket_negative,0


Cleaned shape: (181185, 96)


## 6. Time-Based Train/Test Split

This is not a random split. The test set is the most recent month of available closed sales. The training set is the `X` months immediately preceding that test month.

Business implication: this evaluates whether the model can support forward-looking pricing decisions under the newest market conditions.

In [18]:
# Create a YYYY-MM close_month from CloseDate.
# The split is based on actual transaction month, not the file name.
df_clean["close_month"] = pd.to_datetime(df_clean[DATE_COL]).dt.to_period("M").astype(str)

# Most recent observed close month becomes the test set.
test_period = pd.Period(df_clean["close_month"].max(), freq="M")
close_period = pd.PeriodIndex(df_clean["close_month"], freq="M")

# Show candidate training windows so X is transparent and tunable.
# This does not choose the optimal X by itself; model validation in Week 4 should compare performance.
split_summary_rows = []
for window in TRAINING_WINDOW_CANDIDATES:
    candidate_train_end = test_period - 1
    candidate_train_start = test_period - window
    candidate_train_mask = (close_period >= candidate_train_start) & (close_period <= candidate_train_end)
    candidate_test_mask = close_period == test_period

    candidate_train_rows = int(candidate_train_mask.sum())
    candidate_test_rows = int(candidate_test_mask.sum())
    split_summary_rows.append({
        "test_month": str(test_period),
        "train_start_month": str(candidate_train_start),
        "train_end_month": str(candidate_train_end),
        "training_window_months": window,
        "train_rows": candidate_train_rows,
        "test_rows": candidate_test_rows,
        "excluded_rows_outside_window": len(df_clean) - candidate_train_rows - candidate_test_rows,
        "candidate_status": "valid" if candidate_train_rows > 0 and candidate_test_rows > 0 else "invalid",
    })

split_summary = pd.DataFrame(split_summary_rows)
display(split_summary)
split_summary.to_csv(OUTPUT_DIR / "split_window_candidate_summary.csv", index=False)

# Default split: X months immediately before the test month.
# With X = 12 and test = 2026-05, train = 2025-05 through 2026-04.
train_end_period = test_period - 1
train_start_period = test_period - TRAINING_WINDOW_MONTHS
train_mask = (close_period >= train_start_period) & (close_period <= train_end_period)
test_mask = close_period == test_period

train_df = df_clean[train_mask].copy()
test_df = df_clean[test_mask].copy()

# Store split metadata for the final export.
split_info = {
    "test_month": str(test_period),
    "train_start_month": str(train_start_period),
    "train_end_month": str(train_end_period),
    "training_window_months": TRAINING_WINDOW_MONTHS,
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "excluded_rows_outside_window": len(df_clean) - len(train_df) - len(test_df),
}

print(split_info)
print("Train close months:", train_df["close_month"].min(), "to", train_df["close_month"].max())
print("Test close month:", test_df["close_month"].unique())

,test_month,train_start_month,train_end_month,training_window_months,train_rows,test_rows,excluded_rows_outside_window,candidate_status
0,2026-05,2026-02,2026-04,3,31708,12012,137465,valid
1,2026-05,2025-11,2026-04,6,59327,12012,109846,valid
2,2026-05,2025-08,2026-04,9,94206,12012,74967,valid
3,2026-05,2025-05,2026-04,12,129745,12012,39428,valid
4,2026-05,2025-02,2026-04,15,161045,12012,8128,valid
5,2026-05,2025-01,2026-04,16,169173,12012,0,valid


{'test_month': '2026-05', 'train_start_month': '2025-05', 'train_end_month': '2026-04', 'training_window_months': 12, 'train_rows': 129745, 'test_rows': 12012, 'excluded_rows_outside_window': 39428}
Train close months: 2025-05 to 2026-04
Test close month: ['2026-05']


We use a **12-month training window** as the default because it balances sample size and market recency. Shorter windows may be more current but less stable, while longer windows may include older market conditions that are less relevant to the May 2026 test month. The optimal value of X should be confirmed during model evaluation using R², MAPE, and MdAPE.

## 7. Train-Only Imputation, Encoding, and Normalization

Categorical strategy:

- Low-cardinality categories are one-hot encoded with the first category dropped for linear-model compatibility.
- Geographic/high-cardinality categories are frequency encoded using only the training set, preserving location signal without creating hundreds of sparse columns.
- Names, emails, addresses, and office/person identifiers are not used as model features because they do not create a stable stakeholder decision rule.

Numerical strategy:

- Median imputation is fit on train only.
- Missingness flags are added so the model can learn whether missingness itself has signal.
- Standardized columns are created for linear-regression baseline use; tree models can use the unscaled numeric columns.

In [19]:
# Start from the time-split data.
# Important: all fitted values below must come from train only.
train_model_ready = train_df.copy()
test_model_ready = test_df.copy()

# Store train-fitted preprocessing values.
# This makes it clear which medians, category maps, and scaling values were learned from train.
params = {
    "numeric_medians": {},
    "categorical_modes": {},
    "frequency_maps": {},
    "onehot_categories": {},
    "scaling": {},
}

numeric_cols = []
excluded_all_missing_numeric_cols = []

# Decide which numeric columns can actually be used.
# A numeric column that is 100% missing in train cannot be imputed meaningfully,
# so it is excluded from model-ready features and documented.
for col in KEY_NUMERIC_COLS:
    if col in train_model_ready.columns and col != TARGET_COL:
        non_missing_count = pd.to_numeric(train_model_ready[col], errors="coerce").notna().sum()
        if non_missing_count > 0:
            numeric_cols.append(col)
        else:
            excluded_all_missing_numeric_cols.append(col)

print("Numeric columns used for median imputation and scaling:", numeric_cols)
print("All-missing numeric columns excluded from model-ready features:", excluded_all_missing_numeric_cols)

# Numeric imputation:
# 1. Fit median on train.
# 2. Add a missingness flag to train and test.
# 3. Fill train and test using the train median.
for col in numeric_cols:
    train_median = pd.to_numeric(train_model_ready[col], errors="coerce").median()
    params["numeric_medians"][col] = train_median

    train_model_ready[f"{col}_was_missing"] = train_model_ready[col].isna().astype(int)
    test_model_ready[f"{col}_was_missing"] = test_model_ready[col].isna().astype(int)

    train_model_ready[col] = pd.to_numeric(train_model_ready[col], errors="coerce").fillna(train_median)
    test_model_ready[col] = pd.to_numeric(test_model_ready[col], errors="coerce").fillna(train_median)

# Categorical missing handling:
# Fill missing categorical values using the train mode.
# This avoids using information from the test month.
for col in FREQUENCY_CATEGORICAL_COLS + LOW_CARDINALITY_CATEGORICAL_COLS:
    if col in train_model_ready.columns:
        mode_values = train_model_ready[col].astype("string").fillna("Unknown").mode()
        mode_value = mode_values.iloc[0] if len(mode_values) else "Unknown"
        params["categorical_modes"][col] = mode_value

        train_model_ready[col] = train_model_ready[col].astype("string").fillna(mode_value).replace({"": mode_value})
        test_model_ready[col] = test_model_ready[col].astype("string").fillna(mode_value).replace({"": mode_value})

# Frequency encoding for high-cardinality geographic categories:
# Example: City_frequency = share of training rows in that city.
# This keeps location signal without creating hundreds of dummy variables.
for col in FREQUENCY_CATEGORICAL_COLS:
    if col in train_model_ready.columns:
        frequency_map = (train_model_ready[col].astype("string").fillna("Unknown").value_counts() / len(train_model_ready)).to_dict()
        params["frequency_maps"][col] = frequency_map

        train_model_ready[f"{col}_frequency"] = train_model_ready[col].astype("string").fillna("Unknown").map(frequency_map).fillna(0)
        test_model_ready[f"{col}_frequency"] = test_model_ready[col].astype("string").fillna("Unknown").map(frequency_map).fillna(0)

# One-hot encoding for low-cardinality categories:
# These fields have a small number of values, so dummy variables are readable and manageable.
# The first category is dropped to reduce collinearity for the linear-regression baseline.
for col in LOW_CARDINALITY_CATEGORICAL_COLS:
    if col in train_model_ready.columns:
        categories = sorted(train_model_ready[col].astype("string").fillna("Unknown").unique().tolist())
        params["onehot_categories"][col] = categories

        train_values = train_model_ready[col].astype("string").fillna("Unknown")
        test_values = test_model_ready[col].astype("string").fillna("Unknown")

        for category in categories[1:]:  # drop first category to reduce collinearity for linear baseline
            clean_category = re.sub(r"[^0-9A-Za-z]+", "_", str(category)).strip("_") or "Unknown"
            train_model_ready[f"{col}_is_{clean_category}"] = (train_values == category).astype(int)
            test_model_ready[f"{col}_is_{clean_category}"] = (test_values == category).astype(int)

# Scaling:
# Fit mean/std on train only.
# Scaled columns are useful for the linear-regression baseline.
# Tree-based models can still use the original numeric columns.
for col in numeric_cols:
    train_values = pd.to_numeric(train_model_ready[col], errors="coerce").fillna(params["numeric_medians"].get(col, 0))
    train_mean = train_values.mean()
    train_std = train_values.std(ddof=0)
    if not train_std or np.isnan(train_std):
        train_std = 1.0

    params["scaling"][col] = {"mean": train_mean, "std": train_std}
    train_model_ready[f"{col}_scaled"] = (pd.to_numeric(train_model_ready[col], errors="coerce") - train_mean) / train_std
    test_model_ready[f"{col}_scaled"] = (pd.to_numeric(test_model_ready[col], errors="coerce") - train_mean) / train_std

print("Train model-ready shape:", train_model_ready.shape)
print("Test model-ready shape:", test_model_ready.shape)

Numeric columns used for median imputation and scaling: ['ListPrice', 'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'YearBuilt', 'DaysOnMarket', 'GarageSpaces', 'ParkingTotal', 'Latitude', 'Longitude', 'AssociationFee', 'Stories']
All-missing numeric columns excluded from model-ready features: ['FireplacesTotal']
Train model-ready shape: (129745, 152)
Test model-ready shape: (12012, 152)


## 8. Export Deliverables

Two cleaned outputs are exported:

- `quality_cleaned`: all scoped rows after duplicate handling, strict numerical cleaning, binary conversion, and flags. This preserves all months for audit.
- `model_ready`: only the selected train/test window after train-only imputation, categorical encoding, and scaling. This is the correct modeling input.

In [20]:
# File names are explicit so the team can tell which artifact is for auditing
# and which artifact is ready for modeling.
quality_cleaned_csv = OUTPUT_DIR / "crmls_sfr_quality_cleaned_202501_202605.csv"
model_ready_combined_csv = OUTPUT_DIR / f"crmls_sfr_model_ready_X{TRAINING_WINDOW_MONTHS}_train_test_combined.csv"
train_csv = OUTPUT_DIR / f"crmls_sfr_train_X{TRAINING_WINDOW_MONTHS}_{split_info['train_start_month']}_to_{split_info['train_end_month']}.csv"
test_csv = OUTPUT_DIR / f"crmls_sfr_test_{split_info['test_month']}.csv"
metadata_csv = OUTPUT_DIR / "preprocessing_metadata.csv"

# Add a split label before combining train and test.
# This combined file is convenient for Tableau checks or downstream modeling scripts.
train_model_ready["split"] = "train"
test_model_ready["split"] = "test"
model_ready_combined = pd.concat([train_model_ready, test_model_ready], ignore_index=True, sort=False)

# Missing-value checks:
# quality_cleaned_missing shows what is still missing in the audit dataset.
# model_ready_missing verifies that engineered model-ready columns have no missing values.
quality_cleaned_feature_check = [c for c in KEY_NUMERIC_COLS + FREQUENCY_CATEGORICAL_COLS + LOW_CARDINALITY_CATEGORICAL_COLS if c in df_clean.columns]
model_ready_feature_check = [
    c for c in model_ready_combined.columns
    if c.endswith("_scaled") or c.endswith("_frequency") or c.endswith("_was_missing") or "_is_" in c
]

quality_cleaned_missing = df_clean[quality_cleaned_feature_check].isna().sum().sort_values(ascending=False)
model_ready_missing = model_ready_combined[model_ready_feature_check].isna().sum().sort_values(ascending=False)

quality_cleaned_missing.to_csv(OUTPUT_DIR / "quality_cleaned_missing_check.csv")
model_ready_missing.to_csv(OUTPUT_DIR / "model_ready_missing_check.csv")

# Export the actual deliverables.
df_clean.to_csv(quality_cleaned_csv, index=False)
model_ready_combined.to_csv(model_ready_combined_csv, index=False)
train_model_ready.to_csv(train_csv, index=False)
test_model_ready.to_csv(test_csv, index=False)

# Metadata is a compact audit trail for reviewers.
metadata = pd.DataFrame([
    {"item": "target", "value": TARGET_COL},
    {"item": "filter", "value": 'PropertyType == "Residential" and PropertySubType == "SingleFamilyResidence"'},
    {"item": "test_month", "value": split_info["test_month"]},
    {"item": "train_window_months", "value": split_info["training_window_months"]},
    {"item": "train_months", "value": f"{split_info['train_start_month']} to {split_info['train_end_month']}"},
    {"item": "train_rows", "value": split_info["train_rows"]},
    {"item": "test_rows", "value": split_info["test_rows"]},
    {"item": "excluded_rows_outside_window", "value": split_info["excluded_rows_outside_window"]},
    {"item": "all_missing_numeric_excluded_from_model_ready", "value": ", ".join(excluded_all_missing_numeric_cols) if excluded_all_missing_numeric_cols else "none"},
    {"item": "model_ready_engineered_feature_max_missing", "value": int(model_ready_missing.max()) if len(model_ready_missing) else 0},
    {"item": "quality_cleaned_csv", "value": quality_cleaned_csv.name},
    {"item": "model_ready_combined_csv", "value": model_ready_combined_csv.name},
    {"item": "decision_implication", "value": "Model evaluation simulates pricing decisions on the newest available closed-sale month using only immediately prior months for training."},
])
metadata.to_csv(metadata_csv, index=False)

print("Wrote:")
print(quality_cleaned_csv)
print(model_ready_combined_csv)
print(train_csv)
print(test_csv)
print(metadata_csv)

Wrote:
/content/drive/MyDrive/IDX Summer Intern/outputs/crmls_sfr_quality_cleaned_202501_202605.csv
/content/drive/MyDrive/IDX Summer Intern/outputs/crmls_sfr_model_ready_X12_train_test_combined.csv
/content/drive/MyDrive/IDX Summer Intern/outputs/crmls_sfr_train_X12_2025-05_to_2026-04.csv
/content/drive/MyDrive/IDX Summer Intern/outputs/crmls_sfr_test_2026-05.csv
/content/drive/MyDrive/IDX Summer Intern/outputs/preprocessing_metadata.csv


## Limitations

- The notebook creates a defensible split and preprocessing artifacts; it does not claim the optimal value of `X` without model validation.
- Frequency encoding captures category prevalence, not causal location effects.
- Missing coordinates are removed to support geographic decision use cases, which may underrepresent records with poor geocoding.
- Review outliers remain in the data and must be checked during model error analysis by price segment.

## Decision Implication

The exported train/test files let the team evaluate pricing models on the newest available market month while training only on prior months. That is the right evaluation structure for listing-pricing decisions because it mimics how the model would be used prospectively.